In [1]:
import sys
import os
import json
from pathlib import Path
import importlib
import shutil

import numpy as np
import matplotlib.pyplot as plt
import torch

# Optional but very useful in notebooks
%load_ext autoreload
%autoreload 2

repo_root = Path("/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.supervised_learning.dynamic_deephit_evaluation as dde

DD_RESULTS_DIR = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis"

# pick best model

In [8]:
# i've run multiple seeds for a model trained on data restricted to sepsis patients, while 
# removing all patients in NICU or 19 and over
# sepsis_picu_dir = os.path.join(DD_RESULTS_DIR, 'sepsis_picu')
sepsis_picu_dir = os.path.join(DD_RESULTS_DIR, 'picu')

# find folder in sepsis_picu folder that begins 'choc'
parent_folders = [
    entry.path for entry in os.scandir(sepsis_picu_dir)
    if entry.is_dir() and entry.name.startswith("choc")
]

model_folders = []
for parent_folder in parent_folders:
    # find folders that begin local or unknown
    model_folders.append([
        entry.path for entry in os.scandir(parent_folder)
        if entry.is_dir() and (entry.name.startswith("local")
                               or entry.name.startswith("unknown"))
        ]
    )
# flatten the list of lists
model_folders = [folder for sublist in model_folders for folder in sublist]
model_folders = sorted(model_folders, key=lambda x: os.path.basename(x))


model_losses = {}
for model_folder in model_folders:
    model_name = os.path.basename(model_folder)
    print(f"Evaluating model: {model_name}")

    # load results.json from model_folder
    results_json_path = os.path.join(model_folder, "results.json")
    if not os.path.exists(results_json_path):
        print(f"Results file not found for model {model_name}, skipping.")
        continue
    # load results.json
    with open(results_json_path, "r") as f:
        results = json.load(f)
    model_losses[model_folder] = results['val_scores']['val/weighted_multi_C'][0]


Evaluating model: unknown_cx3-20-1_20260821-202508_49e52257
Evaluating model: unknown_cx3-20-1_20260821-205024_a1047e26
Evaluating model: unknown_cx3-20-1_20260821-210912_95fa0d18
Evaluating model: unknown_cx3-20-1_20260821-213551_0b70bb3a
Evaluating model: unknown_cx3-20-1_20260821-214743_574d96a7


In [9]:
model_losses

{'/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_4_20260821_202242_593682_job3875805.pbs-7/unknown_cx3-20-1_20260821-202508_49e52257': 0.6674920614943447,
 '/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_4_20260821_202242_593682_job3875805.pbs-7/unknown_cx3-20-1_20260821-205024_a1047e26': 0.6521224135375885,
 '/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_4_20260821_202242_593682_job3875805.pbs-7/unknown_cx3-20-1_20260821-210912_95fa0d18': 0.6544045324785164,
 '/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/choc_d

In [10]:
# find the model with the highest "loss", which is actually the highest weighted_multi_C score
best_model_folder = max(model_losses, key=model_losses.get)
best_model_folder

'/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_4_20260821_202242_593682_job3875805.pbs-7/unknown_cx3-20-1_20260821-202508_49e52257'

In [11]:
best_model_folder_base = os.path.basename(best_model_folder)
best_model_folder_base

'unknown_cx3-20-1_20260821-202508_49e52257'

In [12]:
final_model_dir = os.path.join(sepsis_picu_dir, 'final_model', best_model_folder_base)
final_model_dir

'/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model/unknown_cx3-20-1_20260821-202508_49e52257'

In [13]:
# copy the best model folder to the new location
shutil.copytree(best_model_folder, final_model_dir, dirs_exist_ok=True)

# also copy config_manager from best_model_folder's parent directory to the new location
config_manager_src = os.path.join(os.path.dirname(best_model_folder), 'config_manager.pkl')
config_manager_dst = os.path.join(final_model_dir, 'config_manager.pkl')
shutil.copy2(config_manager_src, config_manager_dst)


'/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model/unknown_cx3-20-1_20260821-202508_49e52257/config_manager.pkl'